In [1]:
# 06_actuarial_projection.ipynb
# Illustrative actuarial projection (paper Section 4.7): risk-tier expected-
# cost gradient (Table 12) and recourse cost-offset under the cohort outcomes
# (Table 13). This is an ORDER-OF-MAGNITUDE sizing exercise, NOT a pricing
# model, and every number here is built from EXTERNALLY published cost
# parameters applied to the cohort's transition rates.
#
# ===========================================================================
# LOAD-BEARING CAVEAT - CROSS-SECTIONAL DATA
# ---------------------------------------------------------------------------
# KNHANES is a repeated CROSS-SECTIONAL survey: each annual cycle is an
# independent probability sample and there is NO personal key that links a
# respondent from one year to the next. Consequently:
#   (1) KNHANES contains no claims or expenditure data;
#   (2) a model-predicted tier transition (e.g. Class 3 -> Class 0) is a
#       within-classifier, cross-sectional statement, NOT an observed change
#       in the same person over time;
#   (3) the cost offset below therefore treats a predicted transition AS IF it
#       were realized in claims, which cannot be validated on these data.
# Converting this projection into an underwriting-grade estimate requires
# LONGITUDINAL claims linkage, which KNHANES cannot provide. The figures are
# printed only to express the framework's output in units an insurer uses.
# ===========================================================================
#
# Inputs  (../results/tables/): cohort_results.csv   (transition rates, NB 04)
# Outputs (../results/tables/): table12_cost_gradient.tex,
#                     table13_cost_offset.tex, actuarial_projection.json

import os
import json
import pandas as pd

TAB_DIR = os.path.join("..", "results", "tables")

# ---------------------------------------------------------------------------
# External cost parameters (from published literature - NOT KNHANES-derived)
# ---------------------------------------------------------------------------
# Relative expected-cost multiples vs Class 0, from French et al. (2005):
#   diabetes-only  ~1.59x ; diabetes+hypertension ~2.56x
#   hypertension-only: conservative 1.35x (Wang et al. 2017)
COST_INDEX = {0: 1.00, 1: 1.35, 2: 1.59, 3: 2.56}
# Absolute anchor: Korea NHIS per-capita cost for diabetes = USD 4,090
# (Oh et al. 2021) pins the Class 2 tier.
NHIS_DM_PERCAPITA_USD = 4090.0
CLASS0_ANCHOR_USD = NHIS_DM_PERCAPITA_USD / COST_INDEX[2]   # implied Class 0

# ---------------------------------------------------------------------------
# Table 12 - risk-tier expected-cost gradient
# ---------------------------------------------------------------------------
tier_names = {0: "Class 0 (Normal)", 1: "Class 1 (HTN-only)",
              2: "Class 2 (DM-only)", 3: "Class 3 (Comorbid)"}
cost_rows = []
for k in [0, 1, 2, 3]:
    annual = CLASS0_ANCHOR_USD * COST_INDEX[k]
    excess = annual - CLASS0_ANCHOR_USD
    cost_rows.append({"tier": tier_names[k], "cost_index": COST_INDEX[k],
                      "annual_usd": round(annual), "excess_usd": round(excess)})
cost_df = pd.DataFrame(cost_rows)
print("Table 12 - risk-tier expected-cost gradient (illustrative):")
print(cost_df.to_string(index=False))

E = {k: CLASS0_ANCHOR_USD * COST_INDEX[k] - CLASS0_ANCHOR_USD for k in [0, 1, 2, 3]}

# ---------------------------------------------------------------------------
# Transition rates from the cohort (notebook 04)
# ---------------------------------------------------------------------------
cohort_all = pd.read_csv(os.path.join(TAB_DIR, "experiment_cohort.csv"))
# Use the pre-injection (C2) condition at aggressive delegation for transition rates.
cohort = cohort_all[(cohort_all["cond"] == "C2_pre_rule") &
                    (cohort_all["deleg"] == "aggressive")].reset_index(drop=True)

def transition_probs(df_gender):
    n = len(df_gender); feas = df_gender[df_gender["feasible"]]
    return {"pC0": (feas["achieved_class"] == 0).sum() / n,
            "pC1": (feas["achieved_class"] == 1).sum() / n,
            "pC2": (feas["achieved_class"] == 2).sum() / n,
            "pinf": (n - df_gender["feasible"].sum()) / n}

# ---------------------------------------------------------------------------
# Table 13 - expected gross annual cost offset per Class 3 policyholder
#   E[dC] = a * [ pC0*E3 + pC1*(E3 - E1) + pC2*(E3 - E2) + pinf*0 ]
# evaluated at three adherence levels a in {1.0, 0.5, 0.3}
# ---------------------------------------------------------------------------
def expected_offset(p, a):
    return a * (p["pC0"] * E[3] + p["pC1"] * (E[3] - E[1])
                + p["pC2"] * (E[3] - E[2]))

offset_rows = []
probs_by_gender = {}
for gender in ["Male", "Female"]:
    dfg = cohort[cohort["gender"] == gender]
    if len(dfg) == 0:
        continue
    p = transition_probs(dfg)
    probs_by_gender[gender] = p
    for a, lbl in [(1.0, "Full (100%)"), (0.5, "Moderate (50%)"), (0.3, "Low (30%)")]:
        offset_rows.append({"gender": gender, "adherence": lbl,
                            "offset_usd": round(expected_offset(p, a))})
offset_df = pd.DataFrame(offset_rows)
print("\nTable 13 - projected gross annual cost offset per Class 3 policyholder:")
print(offset_df.to_string(index=False))

# ---------------------------------------------------------------------------
# LaTeX tables
# ---------------------------------------------------------------------------
t12 = [r"\begin{table}[ht]", r"\centering",
       r"\caption{Illustrative risk-tier expected-cost gradient. The relative "
       r"index is anchored to Class~0 using excess-expenditure multiples from "
       r"French et al.; the absolute scale is pinned to the Korea NHIS diabetes "
       r"per-capita cost of USD~4{,}090. Figures are an external-parameter "
       r"projection, not KNHANES-derived estimates.}",
       r"\label{tab:cost_gradient}",
       r"\begin{tabular}{lccc}", r"\toprule",
       r"Risk tier & Cost index & Implied annual cost (USD) & Excess vs. Class 0 (USD) \\",
       r"\midrule"]
for r in cost_rows:
    ex = "--" if r["excess_usd"] == 0 else f"{r['excess_usd']:,}"
    t12.append(f"  {r['tier']} & {r['cost_index']:.2f} & {r['annual_usd']:,} & {ex} \\\\")
t12 += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
with open(os.path.join(TAB_DIR, "table12_cost_gradient.tex"), "w") as f:
    f.write("\n".join(t12))

t13 = [r"\begin{table}[ht]", r"\centering",
       r"\caption{Projected gross annual cost offset per model-confirmed "
       r"Class~3 policyholder, evaluated at three adherence levels. Values are "
       r"illustrative, derived from external cost parameters and the cohort "
       r"transition rates; they are not KNHANES-based cost estimates and, "
       r"because KNHANES is cross-sectional, cannot be validated as realized "
       r"claims offsets.}", r"\label{tab:cost_offset}",
       r"\begin{tabular}{lcc}", r"\toprule",
       r"Adherence $a$ & Male offset (USD) & Female offset (USD) \\", r"\midrule"]
for a, lbl in [(1.0, "Full (100\\%)"), (0.5, "Moderate (50\\%)"), (0.3, "Low (30\\%)")]:
    m = probs_by_gender.get("Male");   f_ = probs_by_gender.get("Female")
    mv = f"{round(expected_offset(m, a)):,}" if m else "--"
    fv = f"{round(expected_offset(f_, a)):,}" if f_ else "--"
    t13.append(f"  {lbl} & {mv} & {fv} \\\\")
t13 += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
with open(os.path.join(TAB_DIR, "table13_cost_offset.tex"), "w") as f:
    f.write("\n".join(t13))

with open(os.path.join(TAB_DIR, "actuarial_projection.json"), "w", encoding="utf-8") as f:
    json.dump({"caveat": "Cross-sectional KNHANES; no claims linkage; "
                         "illustrative order-of-magnitude sizing only.",
               "cost_index": COST_INDEX,
               "class0_anchor_usd": round(CLASS0_ANCHOR_USD),
               "excess_by_tier_usd": {str(k): round(v) for k, v in E.items()},
               "transition_probs": probs_by_gender,
               "cost_gradient": cost_rows, "cost_offset": offset_rows},
              f, ensure_ascii=False, indent=2)

print("\nsaved -> table12_cost_gradient.tex, table13_cost_offset.tex, "
      "actuarial_projection.json")
print("\nREMINDER: cross-sectional data - these are illustrative sizing "
      "figures, not validated pricing or realized claims offsets.")


Table 12 - risk-tier expected-cost gradient (illustrative):
              tier  cost_index  annual_usd  excess_usd
  Class 0 (Normal)        1.00        2572           0
Class 1 (HTN-only)        1.35        3473         900
 Class 2 (DM-only)        1.59        4090        1518
Class 3 (Comorbid)        2.56        6585        4013

Table 13 - projected gross annual cost offset per Class 3 policyholder:
gender      adherence  offset_usd
  Male    Full (100%)        3865
  Male Moderate (50%)        1933
  Male      Low (30%)        1160
Female    Full (100%)        3958
Female Moderate (50%)        1979
Female      Low (30%)        1187

saved -> table12_cost_gradient.tex, table13_cost_offset.tex, actuarial_projection.json

REMINDER: cross-sectional data - these are illustrative sizing figures, not validated pricing or realized claims offsets.
